# Ozon Counterfeit: All Multimodal Experiments

Unified self-contained notebook for modality ablations, multimodal combinations, fusion variants, split validation, and result consistency checks.

This notebook intentionally does not modify existing notebooks, `src/*`, or `scripts/*`.


## Configuration

In [1]:
import os
from pathlib import Path

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

PROJECT_ROOT = Path.cwd().resolve()

TRAIN_CSV = PROJECT_ROOT / 'data/ml_ozon_сounterfeit_data/ml_ozon_сounterfeit_train.csv'
TRAIN_IMAGES_DIR = PROJECT_ROOT / 'data/ml_ozon_сounterfeit_train_images'
MANIFEST_PATH = PROJECT_ROOT / 'artifacts/manifests/train_manifest.csv'

EXPERIMENT_NAMESPACE = 'all_modalities_efficientnet_b0'
IMAGE_MODEL_NAME = 'efficientnet_b0'
TEXT_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
SELECTION_METRIC = 'pr_auc'

# First full run: True. Later reuse run: False.
FORCE_RETRAIN = True

# Smoke run: set to 1000. Full run: None.
MAX_SAMPLES_PER_SPLIT = None

DEVICE = 'mps'
BATCH_SIZE = 256
IMAGE_BATCH_SIZE = max(8, BATCH_SIZE // 4)
NUM_WORKERS = 0

EPOCHS = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 3
PROJECTION_DIM = 256
HIDDEN_DIM = 256
DROPOUT = 0.2

RANDOM_FOREST_ESTIMATORS = 400
LOGISTIC_MAX_ITER = 2000
LOGISTIC_C = 1.0
LATE_FUSION_ALPHAS = None  # Initialized after numpy import.

ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
NAMESPACE_DIR = ARTIFACTS_DIR / 'experiments' / EXPERIMENT_NAMESPACE
CACHE_DIR = NAMESPACE_DIR / 'cache'
REPORTS_DIR = ARTIFACTS_DIR / 'reports'
RESULTS_CSV_PATH = REPORTS_DIR / f'{EXPERIMENT_NAMESPACE}_results.csv'
SPLIT_CHECKS_PATH = REPORTS_DIR / f'{EXPERIMENT_NAMESPACE}_split_checks.json'

PROJECT_ROOT


PosixPath('/Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база')

## Imports

In [2]:
if False:
    %pip install -r requirements.txt


In [3]:
import csv
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ozon_counterfeit.constants import RANDOM_SEED
from ozon_counterfeit.data import load_train_frame, merge_manifest_with_frame
from ozon_counterfeit.encoders import FrozenImageEncoder, FrozenTextEncoder, load_cached_array, save_cached_array
from ozon_counterfeit.features import (
    TabularPreprocessor,
    build_modality_flags,
    build_text_corpus,
    extract_labels,
    infer_log_columns,
    infer_numeric_columns,
)
from ozon_counterfeit.models import EarlyFusionClassifier
from ozon_counterfeit.splitting import compute_split_summary
from ozon_counterfeit.training import NeuralTrainingConfig, choose_threshold_by_f1, compute_metrics
from ozon_counterfeit.utils import ensure_dir, load_json, resolve_torch_device, save_json, set_seed, slugify

LATE_FUSION_ALPHAS = np.linspace(0.0, 1.0, 21)
set_seed(RANDOM_SEED)
print('Imports ready')


Imports ready


## Data Load And Split Checks

In [4]:
def sample_per_split(frame: pd.DataFrame, limit: int) -> pd.DataFrame:
    sampled = []
    for _, split_frame in frame.groupby('split', sort=False):
        take = min(len(split_frame), limit)
        sampled.append(split_frame.sample(n=take, random_state=RANDOM_SEED))
    return pd.concat(sampled, axis=0).sort_values(['split', 'row_id']).reset_index(drop=True)


def load_image_covered_dataset() -> tuple[pd.DataFrame, dict[str, pd.DataFrame], dict[str, object]]:
    train_frame = load_train_frame(TRAIN_CSV)
    manifest_raw = pd.read_csv(MANIFEST_PATH, dtype={'ItemID': str, 'SellerID': str})
    manifest_full = train_frame[['row_id', 'ItemID', 'SellerID', 'resolution']].merge(
        manifest_raw,
        on=['row_id', 'ItemID', 'SellerID', 'resolution'],
        how='inner',
    )
    dataset_full = merge_manifest_with_frame(train_frame, manifest_raw)

    checks: dict[str, object] = {}
    checks['train_rows'] = int(len(train_frame))
    checks['manifest_rows'] = int(len(manifest_full))
    checks['dataset_rows'] = int(len(dataset_full))
    checks['row_count_ok'] = bool(len(manifest_full) == len(train_frame) == len(dataset_full))

    image_rows = dataset_full.loc[dataset_full['has_image'] == 1].copy()
    checks['image_covered_rows'] = int(len(image_rows))
    checks['rows_without_image'] = int((manifest_full['has_image'] == 0).sum())
    checks['all_selected_rows_have_image'] = bool(image_rows['has_image'].eq(1).all())

    exact_mapping_ok = image_rows['image_path'].map(lambda value: Path(value).stem).eq(image_rows['ItemID'].astype(str)).all()
    checks['exact_itemid_image_path_mapping_ok'] = bool(exact_mapping_ok)

    if MAX_SAMPLES_PER_SPLIT:
        image_rows = sample_per_split(image_rows, MAX_SAMPLES_PER_SPLIT)
        checks['max_samples_per_split'] = int(MAX_SAMPLES_PER_SPLIT)
    else:
        checks['max_samples_per_split'] = None

    split_frames = {
        split_name: split_frame.reset_index(drop=True)
        for split_name, split_frame in image_rows.groupby('split', sort=False)
    }
    required_splits = {'train', 'val', 'test'}
    missing_splits = required_splits - set(split_frames)
    checks['required_splits_present'] = bool(not missing_splits)
    checks['missing_splits'] = sorted(missing_splits)
    if missing_splits:
        raise ValueError(f'Missing splits: {sorted(missing_splits)}')

    split_seller_sets = {
        split_name: set(split_frame['SellerID'].astype(str))
        for split_name, split_frame in split_frames.items()
    }
    seller_overlap_counts = {}
    split_names = sorted(split_seller_sets)
    for index, left_name in enumerate(split_names):
        for right_name in split_names[index + 1:]:
            seller_overlap_counts[f'{left_name}__{right_name}'] = len(split_seller_sets[left_name] & split_seller_sets[right_name])
    checks['seller_overlap_counts'] = seller_overlap_counts
    checks['seller_leakage_ok'] = bool(all(count == 0 for count in seller_overlap_counts.values()))

    split_summary = compute_split_summary(image_rows).to_dict(orient='records')
    checks['split_summary'] = split_summary
    checks['class_balance'] = {
        split_name: {
            'rows': int(len(split_frame)),
            'positives': int(split_frame['resolution'].sum()),
            'positive_rate': float(split_frame['resolution'].mean()),
        }
        for split_name, split_frame in split_frames.items()
    }

    critical_checks = [
        checks['row_count_ok'],
        checks['all_selected_rows_have_image'],
        checks['exact_itemid_image_path_mapping_ok'],
        checks['required_splits_present'],
        checks['seller_leakage_ok'],
    ]
    checks['all_critical_checks_ok'] = bool(all(critical_checks))
    if not checks['all_critical_checks_ok']:
        raise ValueError(f'Critical split/data checks failed: {checks}')

    ensure_dir(SPLIT_CHECKS_PATH.parent)
    save_json(SPLIT_CHECKS_PATH, checks)
    return image_rows.reset_index(drop=True), split_frames, checks


dataset, split_frames, split_checks = load_image_covered_dataset()
display(pd.DataFrame(split_checks['split_summary']))
print(f'Split checks saved to: {SPLIT_CHECKS_PATH}')
split_checks


,split,rows,positives,negatives,positive_rate,unique_sellers,with_image_rows
0,test,32563,1957,30606,0.060099,60,32563
1,train,132014,9097,122917,0.068909,11060,132014
2,val,31883,1903,29980,0.059687,34,31883


Split checks saved to: /Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/artifacts/reports/all_modalities_efficientnet_b0_split_checks.json


{'train_rows': 197198,
 'manifest_rows': 197198,
 'dataset_rows': 197198,
 'row_count_ok': True,
 'image_covered_rows': 196460,
 'rows_without_image': 738,
 'all_selected_rows_have_image': True,
 'exact_itemid_image_path_mapping_ok': True,
 'max_samples_per_split': None,
 'required_splits_present': True,
 'missing_splits': [],
 'seller_overlap_counts': {'test__train': 0, 'test__val': 0, 'train__val': 0},
 'seller_leakage_ok': True,
 'split_summary': [{'split': 'test',
   'rows': 32563,
   'positives': 1957,
   'negatives': 30606,
   'positive_rate': 0.060098885237846636,
   'unique_sellers': 60,
   'with_image_rows': 32563},
  {'split': 'train',
   'rows': 132014,
   'positives': 9097,
   'negatives': 122917,
   'positive_rate': 0.0689093580983835,
   'unique_sellers': 11060,
   'with_image_rows': 132014},
  {'split': 'val',
   'rows': 31883,
   'positives': 1903,
   'negatives': 29980,
   'positive_rate': 0.05968698052253552,
   'unique_sellers': 34,
   'with_image_rows': 31883}],
 'c

## Feature Builders And Cache

In [5]:
def save_pickle(path: str | Path, payload: object) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open('wb') as handle:
        pickle.dump(payload, handle)


def load_pickle_local(path: str | Path) -> object:
    with Path(path).open('rb') as handle:
        return pickle.load(handle)


def ensure_finite(name: str, array: np.ndarray) -> None:
    if not np.isfinite(array).all():
        raise ValueError(f'{name} contains NaN or inf values.')


def load_cached_feature(cache_path: Path, expected_rows: int) -> np.ndarray | None:
    if FORCE_RETRAIN:
        return None
    cached = load_cached_array(cache_path)
    if cached is None:
        return None
    if len(cached) != expected_rows:
        print(f'Ignoring stale cache {cache_path}: {len(cached)} rows, expected {expected_rows}')
        return None
    ensure_finite(str(cache_path), cached)
    return cached


def build_or_load_tabular_features(split_frames: dict[str, pd.DataFrame]) -> dict[str, np.ndarray]:
    cache_paths = {split: CACHE_DIR / f'tabular_{split}.npy' for split in ('train', 'val', 'test')}
    preprocessor_path = CACHE_DIR / 'tabular_preprocessor.pkl'
    if not FORCE_RETRAIN and all(path.exists() for path in cache_paths.values()) and preprocessor_path.exists():
        features = {
            split: load_cached_feature(path, len(split_frames[split]))
            for split, path in cache_paths.items()
        }
        if all(value is not None for value in features.values()):
            return {key: value for key, value in features.items() if value is not None}

    numeric_columns = infer_numeric_columns(load_train_frame(TRAIN_CSV))
    log_columns = infer_log_columns(numeric_columns)
    preprocessor = TabularPreprocessor(numeric_columns=numeric_columns, log_columns=log_columns)
    features = {
        'train': preprocessor.fit_transform(split_frames['train']),
        'val': preprocessor.transform(split_frames['val']),
        'test': preprocessor.transform(split_frames['test']),
    }
    for split, array in features.items():
        ensure_finite(f'tabular_{split}', array)
        save_cached_array(cache_paths[split], array)
    save_pickle(preprocessor_path, preprocessor)
    save_json(
        CACHE_DIR / 'tabular_feature_metadata.json',
        {'numeric_columns': numeric_columns, 'log_columns': log_columns, 'feature_dim': int(features['train'].shape[1])},
    )
    return features


def build_or_load_text_features(split_frames: dict[str, pd.DataFrame]) -> dict[str, np.ndarray]:
    model_slug = slugify(TEXT_MODEL_NAME)
    cache_paths = {split: CACHE_DIR / f'text_{split}_{model_slug}.npy' for split in ('train', 'val', 'test')}
    if not FORCE_RETRAIN:
        features = {
            split: load_cached_feature(path, len(split_frames[split]))
            for split, path in cache_paths.items()
        }
        if all(value is not None for value in features.values()):
            return {key: value for key, value in features.items() if value is not None}

    encoder = FrozenTextEncoder(TEXT_MODEL_NAME, device=DEVICE)
    features = {}
    for split in ('train', 'val', 'test'):
        features[split] = encoder.encode(build_text_corpus(split_frames[split]), batch_size=BATCH_SIZE)
        ensure_finite(f'text_{split}', features[split])
        save_cached_array(cache_paths[split], features[split])
    return features


def build_or_load_image_features(split_frames: dict[str, pd.DataFrame]) -> dict[str, np.ndarray]:
    model_slug = slugify(IMAGE_MODEL_NAME)
    cache_paths = {split: CACHE_DIR / f'image_{split}_{model_slug}.npy' for split in ('train', 'val', 'test')}
    if not FORCE_RETRAIN:
        features = {
            split: load_cached_feature(path, len(split_frames[split]))
            for split, path in cache_paths.items()
        }
        if all(value is not None for value in features.values()):
            return {key: value for key, value in features.items() if value is not None}

    encoder = FrozenImageEncoder(IMAGE_MODEL_NAME, device=DEVICE)
    features = {}
    for split in ('train', 'val', 'test'):
        image_paths = split_frames[split]['image_path'].astype(str).tolist()
        features[split] = encoder.encode(image_paths, batch_size=IMAGE_BATCH_SIZE, num_workers=NUM_WORKERS)
        ensure_finite(f'image_{split}', features[split])
        save_cached_array(cache_paths[split], features[split])
    return features


ensure_dir(CACHE_DIR)
labels = {split: extract_labels(split_frames[split]) for split in ('train', 'val', 'test')}
tabular_features = build_or_load_tabular_features(split_frames)
text_features = build_or_load_text_features(split_frames)
image_features = build_or_load_image_features(split_frames)

feature_shapes = pd.DataFrame([
    {'modality': 'tabular', **{split: tabular_features[split].shape for split in ('train', 'val', 'test')}},
    {'modality': 'text', **{split: text_features[split].shape for split in ('train', 'val', 'test')}},
    {'modality': 'image', **{split: image_features[split].shape for split in ('train', 'val', 'test')}},
])
display(feature_shapes)


/Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/src/ozon_counterfeit/encoders.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embedding_dim = self.model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/516 [00:00<?, ?it/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Batches:   0%|          | 0/128 [00:00<?, ?it/s]

Encoding images:   0%|          | 0/2063 [00:00<?, ?it/s]

Encoding images:   0%|          | 0/499 [00:00<?, ?it/s]

Encoding images:   0%|          | 0/509 [00:00<?, ?it/s]

,modality,train,val,test
0,tabular,"(132014, 76)","(31883, 76)","(32563, 76)"
1,text,"(132014, 384)","(31883, 384)","(32563, 384)"
2,image,"(132014, 1280)","(31883, 1280)","(32563, 1280)"


## Metrics And Score Utilities

In [6]:
def experiment_dir(name: str) -> Path:
    return NAMESPACE_DIR / name


def scores_path(name: str) -> Path:
    return experiment_dir(name) / 'scores_val_test.npz'


def has_complete_experiment(name: str, model_filename: str | None = None) -> bool:
    base = experiment_dir(name)
    required = [base / 'metadata.json', base / 'metrics.json', base / 'scores_val_test.npz']
    if model_filename:
        required.append(base / model_filename)
    return all(path.exists() for path in required)


def save_scores(name: str, val_scores: np.ndarray, test_scores: np.ndarray) -> None:
    base = ensure_dir(experiment_dir(name))
    val_scores = val_scores.astype(np.float32)
    test_scores = test_scores.astype(np.float32)
    ensure_finite(f'{name}_val_scores', val_scores)
    ensure_finite(f'{name}_test_scores', test_scores)
    if len(val_scores) != len(labels['val']) or len(test_scores) != len(labels['test']):
        raise ValueError(f'{name} score length mismatch.')
    np.savez_compressed(
        base / 'scores_val_test.npz',
        y_val=labels['val'].astype(np.float32),
        y_test=labels['test'].astype(np.float32),
        val_scores=val_scores,
        test_scores=test_scores,
    )


def load_scores(name: str) -> dict[str, np.ndarray]:
    with np.load(scores_path(name)) as payload:
        return {key: payload[key] for key in payload.files}


def metrics_for_scores(val_scores: np.ndarray, test_scores: np.ndarray) -> dict[str, object]:
    threshold = choose_threshold_by_f1(labels['val'], val_scores)
    return {
        'validation': compute_metrics(labels['val'], val_scores, threshold),
        'test': compute_metrics(labels['test'], test_scores, threshold),
    }


def save_experiment(name: str, metadata: dict[str, object], metrics: dict[str, object]) -> None:
    base = ensure_dir(experiment_dir(name))
    metadata = {
        **metadata,
        'experiment': name,
        'namespace': EXPERIMENT_NAMESPACE,
        'image_model_name': IMAGE_MODEL_NAME,
        'text_model_name': TEXT_MODEL_NAME,
        'selection_metric': SELECTION_METRIC,
        'max_samples_per_split': MAX_SAMPLES_PER_SPLIT,
        'train_rows': int(len(split_frames['train'])),
        'val_rows': int(len(split_frames['val'])),
        'headline_eval_rows': int(len(split_frames['test'])),
    }
    save_json(base / 'metadata.json', metadata)
    save_json(base / 'metrics.json', metrics)


def load_experiment_metrics(name: str) -> dict[str, object]:
    return load_json(experiment_dir(name) / 'metrics.json')


def validate_score_alignment(experiment_names: list[str]) -> dict[str, object]:
    report = {'experiments': experiment_names, 'aligned': True, 'details': {}}
    reference = None
    for name in experiment_names:
        payload = load_scores(name)
        detail = {
            'val_len': int(len(payload['val_scores'])),
            'test_len': int(len(payload['test_scores'])),
            'val_finite': bool(np.isfinite(payload['val_scores']).all()),
            'test_finite': bool(np.isfinite(payload['test_scores']).all()),
        }
        current = (payload['y_val'], payload['y_test'])
        if reference is None:
            reference = current
            detail['label_alignment_ok'] = True
        else:
            detail['label_alignment_ok'] = bool(np.array_equal(reference[0], current[0]) and np.array_equal(reference[1], current[1]))
        if not all([detail['val_finite'], detail['test_finite'], detail['label_alignment_ok']]):
            report['aligned'] = False
        report['details'][name] = detail
    return report


def compact_metric_row(name: str, group: str, metrics: dict[str, object]) -> dict[str, object]:
    val = metrics['validation']
    test = metrics['test']
    return {
        'experiment': name,
        'group': group,
        'val_pr_auc': val['pr_auc'],
        'val_precision': val['precision'],
        'val_recall': val['recall'],
        'val_f1': val['f1'],
        'test_pr_auc': test['pr_auc'],
        'test_precision': test['precision'],
        'test_recall': test['recall'],
        'test_f1': test['f1'],
        'threshold': test['threshold'],
    }


## Simple Modality Experiments

In [7]:
def concat_features(parts: list[dict[str, np.ndarray]]) -> dict[str, np.ndarray]:
    return {
        split: np.hstack([part[split] for part in parts]).astype(np.float32)
        for split in ('train', 'val', 'test')
    }


simple_specs = {
    'tabular_only': {'group': 'single', 'features': ['tabular'], 'model': 'random_forest'},
    'text_only': {'group': 'single', 'features': ['text'], 'model': 'logistic_regression'},
    'image_only': {'group': 'single', 'features': ['image'], 'model': 'logistic_regression'},
    'tabular_text_simple': {'group': 'pair', 'features': ['tabular', 'text'], 'model': 'logistic_regression'},
    'tabular_image_simple': {'group': 'pair', 'features': ['tabular', 'image'], 'model': 'logistic_regression'},
    'text_image_simple': {'group': 'pair', 'features': ['text', 'image'], 'model': 'logistic_regression'},
    'tabular_text_image_simple': {'group': 'triple_simple', 'features': ['tabular', 'text', 'image'], 'model': 'logistic_regression'},
}

feature_registry = {
    'tabular': tabular_features,
    'text': text_features,
    'image': image_features,
}


def build_feature_bundle(feature_names: list[str]) -> dict[str, np.ndarray]:
    return concat_features([feature_registry[name] for name in feature_names])


def train_or_load_simple_experiment(name: str, spec: dict[str, object]) -> dict[str, object]:
    model_filename = 'model.pkl'
    if not FORCE_RETRAIN and has_complete_experiment(name, model_filename=model_filename):
        return {'name': name, 'metrics': load_experiment_metrics(name), 'metadata': load_json(experiment_dir(name) / 'metadata.json')}

    features = build_feature_bundle(spec['features'])
    for split, array in features.items():
        ensure_finite(f'{name}_{split}_features', array)

    if spec['model'] == 'random_forest':
        model = RandomForestClassifier(
            n_estimators=RANDOM_FOREST_ESTIMATORS,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            class_weight='balanced_subsample',
        )
    elif spec['model'] == 'logistic_regression':
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=LOGISTIC_MAX_ITER,
                C=LOGISTIC_C,
                class_weight='balanced',
                solver='saga',
                n_jobs=-1,
                random_state=RANDOM_SEED,
            ),
        )
    else:
        raise ValueError(f'Unknown model type: {spec["model"]}')

    model.fit(features['train'], labels['train'].astype(int))
    val_scores = model.predict_proba(features['val'])[:, 1].astype(np.float32)
    test_scores = model.predict_proba(features['test'])[:, 1].astype(np.float32)
    metrics = metrics_for_scores(val_scores, test_scores)
    metadata = {
        'group': spec['group'],
        'model_type': spec['model'],
        'features': spec['features'],
        'feature_dim': int(features['train'].shape[1]),
    }
    save_pickle(experiment_dir(name) / model_filename, model)
    save_scores(name, val_scores, test_scores)
    save_experiment(name, metadata, metrics)
    return {'name': name, 'metrics': metrics, 'metadata': metadata}


simple_results = []
for name, spec in simple_specs.items():
    result = train_or_load_simple_experiment(name, spec)
    simple_results.append(result)
    display(pd.DataFrame([compact_metric_row(name, spec['group'], result['metrics'])]))


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,tabular_only,single,0.27435,0.438446,0.456647,0.447362,0.224952,0.323248,0.311191,0.317105,0.1025


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,text_only,single,0.738651,0.750136,0.722543,0.736081,0.508607,0.448895,0.715892,0.551792,0.745528


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,image_only,single,0.421412,0.429205,0.610089,0.503906,0.402245,0.465314,0.503832,0.483808,0.697731


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,tabular_text_simple,pair,0.622634,0.716381,0.61587,0.662334,0.494949,0.512195,0.633112,0.566271,0.660466


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,tabular_image_simple,pair,0.321787,0.40835,0.328954,0.364377,0.452993,0.480234,0.50281,0.491263,0.605701


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,text_image_simple,pair,0.727988,0.787146,0.701524,0.741873,0.583662,0.591549,0.622381,0.606574,0.835114


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,tabular_text_image_simple,triple_simple,0.654813,0.679019,0.654756,0.666667,0.557898,0.561693,0.644354,0.60019,0.655293


## Neural Fusion Experiments

In [8]:
class NotebookProjectionBlock(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(input_dim, output_dim), nn.LayerNorm(output_dim), nn.ReLU(), nn.Dropout(dropout))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.layers(inputs)


class GatedFusionClassifier(nn.Module):
    def __init__(self, tabular_dim: int, text_dim: int, image_dim: int, projection_dim: int = 256, hidden_dim: int = 256, dropout: float = 0.2) -> None:
        super().__init__()
        self.tabular_projector = NotebookProjectionBlock(tabular_dim, projection_dim, dropout)
        self.text_projector = NotebookProjectionBlock(text_dim, projection_dim, dropout)
        self.image_projector = NotebookProjectionBlock(image_dim, projection_dim, dropout)
        gate_input_dim = projection_dim * 3 + 3
        self.image_gate = nn.Sequential(
            nn.Linear(gate_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, projection_dim),
            nn.Sigmoid(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(projection_dim * 3 + 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, tabular_features: torch.Tensor, text_features: torch.Tensor, image_features: torch.Tensor, modality_flags: torch.Tensor) -> torch.Tensor:
        tabular_projection = self.tabular_projector(tabular_features)
        text_projection = self.text_projector(text_features)
        image_projection = self.image_projector(image_features)
        gate_inputs = torch.cat([tabular_projection, text_projection, image_projection, modality_flags], dim=1)
        gated_image_projection = image_projection * self.image_gate(gate_inputs)
        fused = torch.cat([tabular_projection, text_projection, gated_image_projection, modality_flags], dim=1)
        return self.classifier(fused).squeeze(1)


class FusionDataset(Dataset):
    def __init__(self, tabular: np.ndarray, text: np.ndarray, image: np.ndarray, flags: np.ndarray, y: np.ndarray) -> None:
        self.tabular = torch.from_numpy(tabular.astype(np.float32))
        self.text = torch.from_numpy(text.astype(np.float32))
        self.image = torch.from_numpy(image.astype(np.float32))
        self.flags = torch.from_numpy(flags.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.float32))

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, index: int):
        return self.tabular[index], self.text[index], self.image[index], self.flags[index], self.y[index]


def make_loader(inputs: dict[str, np.ndarray], batch_size: int, shuffle: bool) -> DataLoader:
    dataset = FusionDataset(inputs['tabular'], inputs['text'], inputs['image'], inputs['flags'], inputs['labels'])
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=False)


def predict_neural_scores(model: nn.Module, loader: DataLoader, device: str, desc: str) -> np.ndarray:
    model.eval()
    scores = []
    iterator = tqdm(loader, desc=desc, leave=False) if tqdm is not None else loader
    with torch.no_grad():
        for tabular, text, image, flags, _ in iterator:
            logits = model(tabular.to(device), text.to(device), image.to(device), flags.to(device))
            scores.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(scores, axis=0).astype(np.float32)


def make_neural_inputs(use_image: bool) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray], dict[str, np.ndarray]]:
    if use_image:
        images = image_features
        image_present = {split: np.ones(len(split_frames[split]), dtype=np.float32) for split in ('train', 'val', 'test')}
    else:
        image_dim = int(image_features['train'].shape[1])
        images = {split: np.zeros((len(split_frames[split]), image_dim), dtype=np.float32) for split in ('train', 'val', 'test')}
        image_present = {split: np.zeros(len(split_frames[split]), dtype=np.float32) for split in ('train', 'val', 'test')}

    packed = {}
    for split in ('train', 'val', 'test'):
        packed[split] = {
            'tabular': tabular_features[split],
            'text': text_features[split],
            'image': images[split],
            'flags': build_modality_flags(len(split_frames[split]), image_present[split]),
            'labels': labels[split],
        }
    return packed['train'], packed['val'], packed['test']


def train_or_load_neural_experiment(name: str, model_kind: str, use_image: bool) -> dict[str, object]:
    if not FORCE_RETRAIN and has_complete_experiment(name, model_filename='model.pt'):
        return {'name': name, 'metrics': load_experiment_metrics(name), 'metadata': load_json(experiment_dir(name) / 'metadata.json')}

    train_inputs, val_inputs, test_inputs = make_neural_inputs(use_image=use_image)
    config = NeuralTrainingConfig(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        patience=PATIENCE,
        projection_dim=PROJECTION_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        random_state=RANDOM_SEED,
        num_workers=NUM_WORKERS,
    )
    set_seed(RANDOM_SEED)
    device = resolve_torch_device(DEVICE)

    model_cls = EarlyFusionClassifier if model_kind == 'early' else GatedFusionClassifier
    model = model_cls(
        tabular_dim=train_inputs['tabular'].shape[1],
        text_dim=train_inputs['text'].shape[1],
        image_dim=train_inputs['image'].shape[1],
        projection_dim=config.projection_dim,
        hidden_dim=config.hidden_dim,
        dropout=config.dropout,
    ).to(device)

    train_loader = make_loader(train_inputs, config.batch_size, shuffle=True)
    val_loader = make_loader(val_inputs, config.batch_size, shuffle=False)
    test_loader = make_loader(test_inputs, config.batch_size, shuffle=False)

    positive_count = float(train_inputs['labels'].sum())
    negative_count = float(len(train_inputs['labels']) - positive_count)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negative_count / max(positive_count, 1.0)], dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

    history = []
    best_pr_auc = float('-inf')
    best_state = None
    best_threshold = 0.5
    epochs_without_improvement = 0
    epoch_iterator = tqdm(range(1, config.epochs + 1), desc=name) if tqdm is not None else range(1, config.epochs + 1)

    for epoch in epoch_iterator:
        model.train()
        running_loss = 0.0
        batch_iterator = tqdm(train_loader, desc=f'{name} epoch {epoch}/{config.epochs}', leave=False) if tqdm is not None else train_loader
        for tabular, text, image, flags, y_batch in batch_iterator:
            optimizer.zero_grad()
            logits = model(tabular.to(device), text.to(device), image.to(device), flags.to(device))
            loss = criterion(logits, y_batch.to(device))
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item()) * len(y_batch)
            if tqdm is not None:
                batch_iterator.set_postfix(loss=f'{float(loss.item()):.4f}')

        train_loss = running_loss / max(len(train_inputs['labels']), 1)
        val_scores = predict_neural_scores(model, val_loader, device, desc=f'{name} validation {epoch}/{config.epochs}')
        val_pr_auc = float(average_precision_score(labels['val'], val_scores))
        val_threshold = choose_threshold_by_f1(labels['val'], val_scores)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_pr_auc': val_pr_auc})

        if val_pr_auc > best_pr_auc:
            best_pr_auc = val_pr_auc
            best_state = {key: value.detach().cpu() for key, value in model.state_dict().items()}
            best_threshold = val_threshold
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if tqdm is not None:
            epoch_iterator.set_postfix(train_loss=f'{train_loss:.4f}', val_pr_auc=f'{val_pr_auc:.4f}', best_pr_auc=f'{best_pr_auc:.4f}')
        if epochs_without_improvement >= config.patience:
            break

    if best_state is None:
        raise RuntimeError(f'{name} did not produce a best checkpoint.')

    model.load_state_dict(best_state)
    val_scores = predict_neural_scores(model, val_loader, device, desc=f'{name} final validation')
    test_scores = predict_neural_scores(model, test_loader, device, desc=f'{name} final test')
    metrics = {
        'validation': compute_metrics(labels['val'], val_scores, best_threshold),
        'test': compute_metrics(labels['test'], test_scores, best_threshold),
    }
    base = ensure_dir(experiment_dir(name))
    torch.save(model.state_dict(), base / 'model.pt')
    with (base / 'training_history.csv').open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['epoch', 'train_loss', 'val_pr_auc'])
        writer.writeheader()
        writer.writerows(history)
    save_scores(name, val_scores, test_scores)
    metadata = {
        'group': 'neural_fusion',
        'model_kind': model_kind,
        'use_image': bool(use_image),
        'tabular_dim': int(train_inputs['tabular'].shape[1]),
        'text_dim': int(train_inputs['text'].shape[1]),
        'image_dim': int(train_inputs['image'].shape[1]),
        'projection_dim': PROJECTION_DIM,
        'hidden_dim': HIDDEN_DIM,
        'dropout': DROPOUT,
        'epochs': EPOCHS,
        'patience': PATIENCE,
    }
    save_experiment(name, metadata, metrics)
    return {'name': name, 'metrics': metrics, 'metadata': metadata}


neural_specs = {
    'tabular_text_early_fusion': {'model_kind': 'early', 'use_image': False},
    'tabular_text_image_early_fusion': {'model_kind': 'early', 'use_image': True},
    'tabular_text_image_gated_fusion': {'model_kind': 'gated', 'use_image': True},
}

neural_results = []
for name, spec in neural_specs.items():
    result = train_or_load_neural_experiment(name, **spec)
    neural_results.append(result)
    display(pd.DataFrame([compact_metric_row(name, 'neural_fusion', result['metrics'])]))


tabular_text_early_fusion:   0%|          | 0/12 [00:00<?, ?it/s]

tabular_text_early_fusion epoch 1/12:   0%|          | 0/516 [00:00<?, ?it/s]

tabular_text_early_fusion validation 1/12:   0%|          | 0/125 [00:00<?, ?it/s]

tabular_text_early_fusion epoch 2/12:   0%|          | 0/516 [00:00<?, ?it/s]

tabular_text_early_fusion validation 2/12:   0%|          | 0/125 [00:00<?, ?it/s]

tabular_text_early_fusion epoch 3/12:   0%|          | 0/516 [00:00<?, ?it/s]

... (54 outputs left out)

tabular_text_image_gated_fusion validation 8/12:   0%|          | 0/125 [00:00<?, ?it/s]

tabular_text_image_gated_fusion final validation:   0%|          | 0/125 [00:00<?, ?it/s]

tabular_text_image_gated_fusion final test:   0%|          | 0/128 [00:00<?, ?it/s]

,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,tabular_text_image_gated_fusion,neural_fusion,0.801312,0.814493,0.738308,0.774531,0.662765,0.645369,0.562596,0.601147,0.739819


## Late Fusion Experiments

In [9]:
all_model_results = simple_results + neural_results


def validation_pr_auc(name: str) -> float:
    return float(load_experiment_metrics(name)['validation']['pr_auc'])


def choose_best(candidates: list[str]) -> str:
    return max(candidates, key=validation_pr_auc)


non_image_candidates = [
    'tabular_only',
    'text_only',
    'tabular_text_simple',
    'tabular_text_early_fusion',
]
image_side_candidates = [
    'image_only',
    'tabular_image_simple',
    'text_image_simple',
]

late_fusion_specs = {
    'late_tabular_text_plus_image_only': ('tabular_text_early_fusion', 'image_only'),
    'late_tabular_text_plus_tabular_image': ('tabular_text_early_fusion', 'tabular_image_simple'),
    'late_best_non_image_plus_best_image_side': (choose_best(non_image_candidates), choose_best(image_side_candidates)),
}


def fit_or_load_late_fusion(name: str, left_name: str, right_name: str) -> dict[str, object]:
    if not FORCE_RETRAIN and has_complete_experiment(name) and (experiment_dir(name) / 'fusion_config.json').exists():
        return {'name': name, 'metrics': load_experiment_metrics(name), 'metadata': load_json(experiment_dir(name) / 'metadata.json')}

    left = load_scores(left_name)
    right = load_scores(right_name)
    if not np.array_equal(left['y_val'], right['y_val']) or not np.array_equal(left['y_test'], right['y_test']):
        raise ValueError(f'Score labels are not aligned for {left_name} and {right_name}.')

    candidates = []
    for alpha in LATE_FUSION_ALPHAS:
        val_scores = alpha * left['val_scores'] + (1.0 - alpha) * right['val_scores']
        threshold = choose_threshold_by_f1(labels['val'], val_scores)
        val_metrics = compute_metrics(labels['val'], val_scores, threshold)
        candidates.append(
            {
                'alpha': float(alpha),
                'threshold': float(threshold),
                'validation_pr_auc': float(val_metrics['pr_auc']),
                'validation_f1': float(val_metrics['f1']),
            }
        )

    best = max(candidates, key=lambda item: (item['validation_pr_auc'], item['validation_f1']))
    val_scores = best['alpha'] * left['val_scores'] + (1.0 - best['alpha']) * right['val_scores']
    test_scores = best['alpha'] * left['test_scores'] + (1.0 - best['alpha']) * right['test_scores']
    metrics = {
        'validation': compute_metrics(labels['val'], val_scores, best['threshold']),
        'test': compute_metrics(labels['test'], test_scores, best['threshold']),
    }
    metadata = {
        'group': 'late_fusion',
        'left_experiment': left_name,
        'right_experiment': right_name,
        'selected_alpha': best['alpha'],
        'selected_threshold': best['threshold'],
        'selected_by': 'validation_pr_auc_then_validation_f1',
    }
    save_scores(name, val_scores, test_scores)
    save_experiment(name, metadata, metrics)
    save_json(experiment_dir(name) / 'fusion_config.json', {'best': best, 'candidates': candidates, **metadata})
    return {'name': name, 'metrics': metrics, 'metadata': metadata}


late_fusion_results = []
for name, (left_name, right_name) in late_fusion_specs.items():
    result = fit_or_load_late_fusion(name, left_name, right_name)
    late_fusion_results.append(result)
    display(pd.DataFrame([compact_metric_row(name, 'late_fusion', result['metrics'])]))


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,late_tabular_text_plus_image_only,late_fusion,0.784414,0.666229,0.800315,0.727143,0.593553,0.572175,0.615738,0.593158,0.49261


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,late_tabular_text_plus_tabular_image,late_fusion,0.784414,0.666229,0.800315,0.727143,0.593553,0.572175,0.615738,0.593158,0.49261


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,late_best_non_image_plus_best_image_side,late_fusion,0.808278,0.793472,0.779296,0.78632,0.630212,0.621562,0.577414,0.598675,0.645928


## Validation And Consistency Checks

In [10]:
all_experiment_names = [result['name'] for result in simple_results + neural_results + late_fusion_results]
score_alignment_report = validate_score_alignment(all_experiment_names)
if not score_alignment_report['aligned']:
    raise ValueError(f'Score alignment failed: {score_alignment_report}')

consistency_report = {
    'split_checks': split_checks,
    'score_alignment': score_alignment_report,
    'feature_shapes': {
        'tabular': {split: list(tabular_features[split].shape) for split in ('train', 'val', 'test')},
        'text': {split: list(text_features[split].shape) for split in ('train', 'val', 'test')},
        'image': {split: list(image_features[split].shape) for split in ('train', 'val', 'test')},
    },
}
save_json(NAMESPACE_DIR / 'consistency_checks.json', consistency_report)
print(f'Consistency checks saved to: {NAMESPACE_DIR / "consistency_checks.json"}')
score_alignment_report


Consistency checks saved to: /Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/artifacts/experiments/all_modalities_efficientnet_b0/consistency_checks.json


{'experiments': ['tabular_only',
  'text_only',
  'image_only',
  'tabular_text_simple',
  'tabular_image_simple',
  'text_image_simple',
  'tabular_text_image_simple',
  'tabular_text_early_fusion',
  'tabular_text_image_early_fusion',
  'tabular_text_image_gated_fusion',
  'late_tabular_text_plus_image_only',
  'late_tabular_text_plus_tabular_image',
  'late_best_non_image_plus_best_image_side'],
 'aligned': True,
 'details': {'tabular_only': {'val_len': 31883,
   'test_len': 32563,
   'val_finite': True,
   'test_finite': True,
   'label_alignment_ok': True},
  'text_only': {'val_len': 31883,
   'test_len': 32563,
   'val_finite': True,
   'test_finite': True,
   'label_alignment_ok': True},
  'image_only': {'val_len': 31883,
   'test_len': 32563,
   'val_finite': True,
   'test_finite': True,
   'label_alignment_ok': True},
  'tabular_text_simple': {'val_len': 31883,
   'test_len': 32563,
   'val_finite': True,
   'test_finite': True,
   'label_alignment_ok': True},
  'tabular_imag

## Final Comparison Table

In [11]:
rows = []
for result in simple_results:
    rows.append(compact_metric_row(result['name'], result['metadata']['group'], result['metrics']))
for result in neural_results:
    rows.append(compact_metric_row(result['name'], result['metadata']['group'], result['metrics']))
for result in late_fusion_results:
    rows.append(compact_metric_row(result['name'], result['metadata']['group'], result['metrics']))

results_table = pd.DataFrame(rows).sort_values(['val_pr_auc', 'val_f1'], ascending=[False, False]).reset_index(drop=True)
ensure_dir(RESULTS_CSV_PATH.parent)
results_table.to_csv(RESULTS_CSV_PATH, index=False)
display(results_table)
print(f'Results saved to: {RESULTS_CSV_PATH}')


,experiment,group,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1,threshold
0,late_best_non_image_plus_best_image_side,late_fusion,0.808278,0.793472,0.779296,0.786320,0.630212,0.621562,0.577414,0.598675,0.645928
1,tabular_text_image_early_fusion,neural_fusion,0.808159,0.795442,0.770363,0.782702,0.606807,0.614045,0.540623,0.575000,0.623433
2,tabular_text_image_gated_fusion,neural_fusion,0.801312,0.814493,0.738308,0.774531,0.662765,0.645369,0.562596,0.601147,0.739819
3,tabular_text_early_fusion,neural_fusion,0.784414,0.666229,0.800315,0.727143,0.593553,0.572175,0.615738,0.593158,0.492610
4,late_tabular_text_plus_image_only,late_fusion,0.784414,0.666229,0.800315,0.727143,0.593553,0.572175,0.615738,0.593158,0.492610
5,late_tabular_text_plus_tabular_image,late_fusion,0.784414,0.666229,0.800315,0.727143,0.593553,0.572175,0.615738,0.593158,0.492610
6,text_only,single,0.738651,0.750136,0.722543,0.736081,0.508607,0.448895,0.715892,0.551792,0.745528
7,text_image_simple,pair,0.727988,0.787146,0.701524,0.741873,0.583662,0.591549,0.622381,0.606574,0.835114
8,tabular_text_image_simple,triple_simple,0.654813,0.679019,0.654756,0.666667,0.557898,0.561693,0.644354,0.600190,0.655293
9,tabular_text_simple,pair,0.622634,0.716381,0.615870,0.662334,0.494949,0.512195,0.633112,0.566271,0.660466


Results saved to: /Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/artifacts/reports/all_modalities_efficientnet_b0_results.csv


## Best Model Summary

In [12]:
best_row = results_table.iloc[0].to_dict()
best_summary = {
    'selected_by': 'validation_pr_auc_then_validation_f1',
    'best_experiment': best_row['experiment'],
    'best_group': best_row['group'],
    'validation_pr_auc': float(best_row['val_pr_auc']),
    'validation_f1': float(best_row['val_f1']),
    'test_pr_auc': float(best_row['test_pr_auc']),
    'test_precision': float(best_row['test_precision']),
    'test_recall': float(best_row['test_recall']),
    'test_f1': float(best_row['test_f1']),
    'threshold': float(best_row['threshold']),
}
save_json(NAMESPACE_DIR / 'best_model_summary.json', best_summary)
best_summary


{'selected_by': 'validation_pr_auc_then_validation_f1',
 'best_experiment': 'late_best_non_image_plus_best_image_side',
 'best_group': 'late_fusion',
 'validation_pr_auc': 0.808278238853294,
 'validation_f1': 0.7863202545068929,
 'test_pr_auc': 0.6302120386185919,
 'test_precision': 0.6215621562156216,
 'test_recall': 0.5774144098109351,
 'test_f1': 0.5986754966887418,
 'threshold': 0.645928218960762}